In [ ]:
%pip -qqq install optuna "optuna-integration[tfkeras]"

In [ ]:
import gc

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import optuna
import tensorflow as tf
import tensorflow.keras.backend as K

from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Masking, LSTM, Dense


In [ ]:
seed = 42

## 1. データの読み込み

In [ ]:
path = "../data/data.csv"

df_raw = pd.read_csv(path)

In [ ]:
df_raw.head(3)

## 2. データの前処理

In [ ]:
df_data = df_raw.copy()

In [ ]:
# 電流、電圧、温度だけを特徴量にする
# BCtはほぼりーくなので、使わない

list_features = ["chI", "chV", "chT", "disI", "disV", "disT"]

y_col = "SOH"

In [ ]:
# 標準化用のインスタンス
scaler = StandardScaler()

In [ ]:
for col in list_features:
    df_data[[col]] = scaler.fit_transform(df_data[[col]])

# 簡易実験のため、train, val, testのデータセットに分ける前に全体で標準化

In [ ]:
df_data

In [ ]:
# バッテリーIDを取り出し
list_battery_ids = df_data["battery_id"].unique()

In [ ]:
list_battery_ids

## 3. ベースモデルの学習

In [ ]:
# データセット作成
#　時系列データなので、ランダム分割ではなく、時系列を保持して分割
# 今回はバッテリーidごとに分割

df_train = df_data[df_data["battery_id"] == list_battery_ids[0]]
df_val = df_data[df_data["battery_id"] == list_battery_ids[1]]
df_test = df_data[df_data["battery_id"] == list_battery_ids[2]]

### 3.1 LSTM

#### 3.1.1 データセット作成

In [ ]:
# dfをlstmの入力ように三次元配列に変換する

def to_lstm_input_data(df_x, df_y, time_steps):

    X_list = []
    y_list = []

    for i in range(len(df_x) - time_steps):

        X_list.append(df_x.iloc[i : i + time_steps].values)
        
        y_list.append(df_y.iloc[i + time_steps])

    return np.array(X_list), np.array(y_list) 

#### 3.1.2 モデル構築

In [ ]:
# モデルに層の追加
unit_num = 32
time_steps = 10
feature_len = len(list_features)

#乱数設定
tf.keras.utils.set_random_seed(seed)

# TensorFlowの「決定論的動作」を設定。最終版のモデルの学習時はコメントアウト
tf.config.experimental.enable_op_determinism()

model_lstm_baseline = Sequential()

model_lstm_baseline.add(LSTM(unit_num, input_shape=(time_steps, feature_len)))
model_lstm_baseline.add(Dense(1))

In [ ]:
model_lstm_baseline.compile(optimizer="adam", loss="mae")
model_lstm_baseline.summary()

#### 3.1.3 モデルの学習

In [ ]:
X_train_lstm, y_train_lstm = to_lstm_input_data(
    df_x=df_train[list_features],
    df_y=df_train[y_col],
    time_steps=time_steps
)

In [ ]:
X_val_lstm, y_val_lstm = to_lstm_input_data(
    df_x=df_val[list_features],
    df_y=df_val[y_col],
    time_steps=time_steps
)

In [ ]:
X_test_lstm, y_test_lstm = to_lstm_input_data(
    df_x=df_test[list_features],
    df_y=df_test[y_col],
    time_steps=time_steps
)

In [ ]:
model_lstm_baseline.fit(X_train_lstm,
               y_train_lstm,
               epochs=5,
               batch_size=16,
               validation_data=(X_val_lstm, y_val_lstm)
               )

## 4. ベースモデルの予測と評価

### 4.1 LSTM

In [ ]:
y_pred_lstm = model_lstm_baseline.predict(X_test_lstm)

In [ ]:
mae_lstm = mean_absolute_error(y_test_lstm, y_pred_lstm)
K.clear_session()
print(mae_lstm)

In [ ]:
y_col

In [ ]:
# maeが59で大きい。
# SOHの平均値に近い？
# グラフで見てみる

x = np.arange(0, len(y_test_lstm))

plt.plot(x, y_test_lstm, label="Actual")
plt.plot(x, y_pred_lstm, label="Predicted")
plt.title("Actual SOH vs Predicted SOH")
plt.legend()
plt.show()


## 5. Model Improvement

In [ ]:
df_feature_engineering = df_data.copy()

### 5.1 特徴量の追加

In [ ]:
# 移動平均の追加
window_ma = 5

df_feature_engineering["chI_ma"] = df_feature_engineering["chI"].rolling(window=window_ma).mean()
df_feature_engineering["chV_ma"] = df_feature_engineering["chV"].rolling(window=window_ma).mean()
df_feature_engineering["chT_ma"] = df_feature_engineering["chT"].rolling(window=window_ma).mean()

df_feature_engineering["disI_ma"] = df_feature_engineering["disI"].rolling(window=window_ma).mean()
df_feature_engineering["disV_ma"] = df_feature_engineering["disV"].rolling(window=window_ma).mean()
df_feature_engineering["disT_ma"] = df_feature_engineering["disT"].rolling(window=window_ma).mean()

In [ ]:
# 傾きを計算する関数
def calc_slope(y):

    list_x = np.arange(len(y))
                       
    slope, _ = np.polyfit(list_x, y, 1)

    return slope

In [ ]:
# 傾きの追加
window_slope = 3

df_feature_engineering["chI_slope"] = df_feature_engineering["chI"].rolling(window=window_slope).apply(calc_slope)
df_feature_engineering["chV_slope"] = df_feature_engineering["chV"].rolling(window=window_slope).apply(calc_slope)
df_feature_engineering["chT_slope"] = df_feature_engineering["chT"].rolling(window=window_slope).apply(calc_slope)

df_feature_engineering["disI_slope"] = df_feature_engineering["disI"].rolling(window=window_slope).apply(calc_slope)
df_feature_engineering["disV_slope"] = df_feature_engineering["disV"].rolling(window=window_slope).apply(calc_slope)
df_feature_engineering["disT_slope"] = df_feature_engineering["disT"].rolling(window=window_slope).apply(calc_slope)

In [ ]:
df_feature_engineering.head(5)

In [ ]:
# 移動平均、傾きを求めたい際のnanは０埋めする
# →lstmでmask layerを追加する

df_feature_engineering = df_feature_engineering.fillna(0)

### 5.2 データセット作成

In [ ]:
def makedataset(df, list_features_imp, y_col, time_steps_imp, list_battery_ids):
    
    df_feature_engineering = df
    df_train_imp = df_feature_engineering[df_feature_engineering["battery_id"] == list_battery_ids[0]]
    df_val_imp = df_feature_engineering[df_feature_engineering["battery_id"] == list_battery_ids[1]]
    df_test_imp = df_feature_engineering[df_feature_engineering["battery_id"] == list_battery_ids[2]]

    X_train_lstm_imp, y_train_lstm_imp = to_lstm_input_data(
        df_x=df_train_imp[list_features_imp],
        df_y=df_train_imp[y_col],
        time_steps=time_steps_imp
    )

    X_val_lstm_imp, y_val_lstm_imp = to_lstm_input_data(
        df_x=df_val_imp[list_features_imp],
        df_y=df_val_imp[y_col],
        time_steps=time_steps_imp
    )

    X_test_lstm_imp, y_test_lstm_imp = to_lstm_input_data(
        df_x=df_test_imp[list_features_imp],
        df_y=df_test_imp[y_col],
        time_steps=time_steps_imp
    )

    return X_train_lstm_imp, y_train_lstm_imp, X_val_lstm_imp, y_val_lstm_imp, X_test_lstm_imp, y_test_lstm_imp
    

### 5.3 学習と評価

In [ ]:
def model_train(unit_num, time_steps_imp, list_features_imp, X_train_lstm_imp, 
                y_train_lstm_imp, X_val_lstm_imp, y_val_lstm_imp, batch_size,epochs
                ):

    features_impr_len = len(list_features_imp)
    input_shape=(time_steps_imp, features_impr_len)

    #乱数設定
    tf.keras.utils.set_random_seed(seed)

    # TensorFlowの「決定論的動作」を設定。最終版のモデルの学習時はコメントアウト
    tf.config.experimental.enable_op_determinism()

    model = Sequential()

    model.add(Masking(mask_value=0, input_shape=input_shape))
    model.add(LSTM(unit_num, input_shape=input_shape))
    model.add(Dense(1))


    model.compile(optimizer="adam", loss="mae")
    # model.summary()

    model.fit(X_train_lstm_imp,
               y_train_lstm_imp,
               epochs=epochs,
               batch_size=batch_size,
               validation_data=(X_val_lstm_imp, y_val_lstm_imp)
               )

    return model

In [ ]:
def model_eval(model, X_test_lstm_imp, y_test_lstm_imp):

    y_pred = model.predict(X_test_lstm_imp)

    mae = mean_absolute_error(y_test_lstm_imp, y_pred)
    
    print("mae:", mae)

    K.clear_session()

    return y_pred, round(mae, 2)

In [ ]:
list_features_imp = list_features + ["chI_ma", "chV_ma", "chT_ma", "disI_ma", 
                                    "disV_ma","disT_ma", "chI_slope", "chV_slope", 
                                    "chT_slope", "disI_slope","disV_slope", "disT_slope"
                                    ]

# list_features_imp = list_features

In [ ]:
time_steps_imp = 30

X_train_lstm_imp, y_train_lstm_imp, X_val_lstm_imp, y_val_lstm_imp, X_test_lstm_imp, y_test_lstm_imp = makedataset(
    df=df_feature_engineering, 
    list_features_imp=list_features_imp, 
    y_col=y_col, 
    time_steps_imp=time_steps_imp, 
    list_battery_ids=list_battery_ids
    )

In [ ]:
unit_num_imp = 512
epochs = 20

model_lstm_imp = model_train(
    unit_num=unit_num_imp,
    time_steps_imp=time_steps_imp, 
    list_features_imp=list_features_imp, 
    X_train_lstm_imp=X_train_lstm_imp, 
    y_train_lstm_imp=y_train_lstm_imp, 
    X_val_lstm_imp=X_val_lstm_imp, 
    y_val_lstm_imp=y_val_lstm_imp,
    batch_size=16,
    epochs=epochs
)

In [ ]:
y_pred_lstm_imp, mae_imp = model_eval(
    model=model_lstm_imp, 
    X_test_lstm_imp=X_test_lstm_imp, 
    y_test_lstm_imp=y_test_lstm_imp
)

In [ ]:
print(f"feature_num,unit_num,time_steps,epochs,mae")
print(f"{len(list_features_imp)},{unit_num_imp},{time_steps_imp},{epochs},{mae_imp}")

## 6. Optunaによる最適化

In [ ]:
def objective(trial):

    time_steps = time_steps_imp #datasetを作った時のglobal変数を指定
    feature_len = len(list_features_imp)
    input_shape=(time_steps, feature_len)

    # 探索空間の定義
    n_units = trial.suggest_int("n_units", 32, 1024, step=32)
    # learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)
    # dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)

    #乱数設定
    tf.keras.utils.set_random_seed(seed)

    # TensorFlowの「決定論的動作」を設定。最終版のモデルの学習時はコメントアウト
    tf.config.experimental.enable_op_determinism()

    model = Sequential()

    model.add(Masking(mask_value=0, input_shape=input_shape))
    model.add(LSTM(n_units, input_shape=input_shape))
    model.add(Dense(1))

    optimizer = tf.keras.optimizers.Adam()
    # optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(optimizer=optimizer, loss="mae")

    early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3)
    pruning_callback = optuna.integration.TFKerasPruningCallback(trial, "val_loss")

    history = model.fit(
        X_train_lstm_imp, y_train_lstm_imp,
        validation_data=(X_val_lstm_imp, y_val_lstm_imp),
        epochs=20,
        batch_size=16,
        callbacks=[early_stop, pruning_callback],
        verbose=0
    )

    val_loss = min(history.history["val_loss"])

    del model               
    K.clear_session() # 今回は学習だけなので、ループ内でメモリを解放

    # Pythonのガベージコレクションを強制実行
    gc.collect()            

    return val_loss

In [ ]:
study = optuna.create_study(direction="minimize", pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=30)

In [ ]:
print("Best hyperparameters: ", study.best_params)
print("Best value: ", study.best_value)

In [ ]:
model_best_n_units = model_train(
    unit_num=864,
    time_steps_imp=time_steps_imp, 
    list_features_imp=list_features_imp, 
    X_train_lstm_imp=X_train_lstm_imp, 
    y_train_lstm_imp=y_train_lstm_imp, 
    X_val_lstm_imp=X_val_lstm_imp, 
    y_val_lstm_imp=y_val_lstm_imp,
    batch_size=16,
    epochs=20
)

In [ ]:
y_pred_best, mae_best = model_eval(
    model=model_best_n_units, 
    X_test_lstm_imp=X_test_lstm_imp, 
    y_test_lstm_imp=y_test_lstm_imp
)